In [1]:
IN_DIR = "/content/outputs/m1_composition"  # adjustable
ZIP_PATH = "/content/m1_composition_outputs.zip"  # adjustable
RESULTS_DIR = "/content/outputs/m1_composition_prompts"  # adjustable
METHOD_ID = "m1_composition"
DRY_RUN = False  # adjustable
MODEL = "models/gemini-3.5-flash"  # adjustable
API_VERSION = "v1"
TEMPERATURE = 1.0  # adjustable
MAX_OUTPUT_TOKENS = 8192
N_RUNS = 5  # adjustable
TEMPLATES = ["T1", "T2", "T3", "T4"]
TISSUE_CONTEXT = "human tumour tissue profiled with NanoString CosMx SMI, 960-gene panel"  # adjustable
TOP_GENES = 10
TOP_STATES = 5
LEGEND_GENES = 5
LETTER_SEED = 7
SLEEP_RANGE = (1, 5)
MAX_RETRIES = 6
TIMEOUT_SEC = 180

In [2]:
import os, re, json, time, random, shutil, zipfile, sys, datetime
import numpy as np, pandas as pd, requests

os.makedirs(f"{RESULTS_DIR}/raw", exist_ok=True); os.makedirs(f"{RESULTS_DIR}/prompts", exist_ok=True)
NEED = ["niche_markers_long.csv", "niche_composition.csv", "state_markers.csv", "niche_summary.csv"]
if not all(os.path.exists(f"{IN_DIR}/{f}") for f in NEED):
    if not os.path.exists(ZIP_PATH):
        from google.colab import files
        ZIP_PATH = "/content/" + list(files.upload())[0]
    os.makedirs(IN_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(IN_DIR)

mk = pd.read_csv(f"{IN_DIR}/niche_markers_long.csv", dtype={"niche": str})
nc = pd.read_csv(f"{IN_DIR}/niche_composition.csv", dtype={"niche": str, "state": str})
sm = pd.read_csv(f"{IN_DIR}/state_markers.csv", dtype={"state": str})
summ = pd.read_csv(f"{IN_DIR}/niche_summary.csv", dtype={"niche": str})
assert {"niche", "rank", "gene", "log2FC"} <= set(mk.columns)
assert {"niche", "state", "frac_in_niche", "fold_enrichment", "rank"} <= set(nc.columns)
assert {"state", "rank", "gene"} <= set(sm.columns)
assert {"niche", "n_cells"} <= set(summ.columns)

ids = sorted(summ["niche"].unique(), key=int)
assert len(ids) <= 26
random.Random(LETTER_SEED).shuffle(ids)
letters = [chr(65 + i) for i in range(len(ids))]
niche_of = dict(zip(letters, ids))
n_cells_of = dict(zip(summ["niche"], summ["n_cells"]))
key = pd.DataFrame({"niche_letter": letters, "niche_id": ids, "n_cells": [int(n_cells_of[i]) for i in ids]})
key.to_csv(f"{RESULTS_DIR}/niche_letter_key.csv", index=False)

sids = sorted(sm["state"].unique(), key=int)
slabel = {s: f"S{i + 1}" for i, s in enumerate(sids)}
assert set(nc["state"]) <= set(slabel)
pd.DataFrame({"state_id": sids, "state_label": [slabel[s] for s in sids]}).to_csv(f"{RESULTS_DIR}/state_legend_key.csv", index=False)
print(len(letters), "regions;", len(sids), "states")

5 regions; 9 states


In [3]:
FORBIDDEN_CI = ["banksy", "myeloid-enriched stroma", "stroma", "tumor interior", "lymphoid structure",
                "tumor-stroma boundary", "neutrophils", "macrophages", "immune"]
FORBIDDEN_CS = ["Region"]

def genes_txt(nid, with_fc):
    d = mk[mk["niche"] == nid].sort_values("rank").head(TOP_GENES)
    return ", ".join(f"{g} ({l:+.2f})" if with_fc else g for g, l in zip(d["gene"], d["log2FC"]))

def states_txt(nid, with_fe):
    d = nc[nc["niche"] == nid].sort_values("rank").head(TOP_STATES)
    return ", ".join(f"{slabel[s]} {100 * p:.1f}%" + (f" (fold-enrichment {fe:.1f})" if with_fe else "")
                     for s, p, fe in zip(d["state"], d["frac_in_niche"], d["fold_enrichment"]))

def legend_txt():
    out = []
    for s in sids:
        g = sm[sm["state"] == s].sort_values("rank").head(LEGEND_GENES)["gene"]
        out.append(f"{slabel[s]}: " + ", ".join(g))
    return "\n".join(out)

def build_prompt(tpl):
    n = len(letters)
    p = ["You are an expert computational biologist and spatial transcriptomics specialist.", "",
         "### Task:", "Identify and annotate the biological cellular niche (tissue microenvironment identity) for each region below.", "",
         "### Metadata:", "- Modality/Platform: NanoString CosMx SMI (960-gene panel)",
         f"- Number of regions: {n} (A-{letters[-1]})", "- Target Species: Human"]
    if tpl != "T1":
        p += [f"- Tissue context: {TISSUE_CONTEXT}", "",
              "### State legend:", "States are anonymous cell-state clusters. Each line gives a state and its top marker genes.", legend_txt()]
    p += ["", "### Marker genes per region:"]
    if tpl == "T3" or tpl == "T4":
        p += ["Genes are ranked top markers with log2 fold-change versus the other regions in parentheses.",
              "Each state share is the percent of the region's cellular neighbourhood in that state, with fold-enrichment over the global share."]
    elif tpl == "T2":
        p += ["Each state share is the percent of the region's cellular neighbourhood in that state."]
    for L in letters:
        nid = niche_of[L]
        p += ["", f"[{L}]", "genes: " + genes_txt(nid, tpl in ("T3", "T4"))]
        if tpl != "T1":
            p.append("top states: " + states_txt(nid, tpl in ("T3", "T4")))
    p += ["", "### Response format:"]
    if tpl == "T4":
        p += ["Reason step by step first: (1) name the dominant cell states of each region from their marker genes, "
              "(2) infer the cell-type composition, (3) infer the niche.", "Then end your reply with the final answer as a fenced JSON block:"]
    else:
        p += ["Reply with a fenced JSON block:"]
    p += ['```json', '[{"region": "A", "label": "...", "confidence": "high|medium|low", "justification": "..."}]', '```',
          f"Include exactly one object for each region A-{letters[-1]}."]
    return "\n".join(p)

PROMPTS = {t: build_prompt(t) for t in TEMPLATES}
for t, p in PROMPTS.items():
    for w in FORBIDDEN_CS: assert w not in p, (t, w)
    for w in FORBIDDEN_CI: assert w not in p.lower(), (t, w)
    open(f"{RESULTS_DIR}/prompts/{t}.txt", "w").write(p)
print("prompt leakage check passed")
if DRY_RUN:
    for t, p in PROMPTS.items():
        print("=" * 30, t, "=" * 30); print(p)

prompt leakage check passed


In [4]:
class StopRun(Exception):
    pass

def call_gemini(prompt, api_key):
    url = f"https://generativelanguage.googleapis.com/{API_VERSION}/{MODEL}:generateContent"
    headers = {"Content-Type": "application/json", "x-goog-api-key": api_key}
    body = {"contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"temperature": TEMPERATURE, "maxOutputTokens": MAX_OUTPUT_TOKENS}}
    for a in range(MAX_RETRIES):
        wait = min(60, 2 ** (a + 1)) + random.random()
        try:
            r = requests.post(url, headers=headers, json=body, timeout=TIMEOUT_SEC)
        except requests.exceptions.RequestException as e:
            print("connection error, retrying:", type(e).__name__); time.sleep(wait); continue
        if r.status_code == 429: raise StopRun("429: " + r.text[:200])
        if r.status_code in (500, 503):
            print(r.status_code, "retrying"); time.sleep(wait); continue
        if r.status_code != 200: raise StopRun(f"{r.status_code}: {r.text[:200]}")
        return r.json()
    raise StopRun("retries exhausted")

def raw_path(tpl, run):
    return f"{RESULTS_DIR}/raw/{METHOD_ID}_{tpl}_run{run}.json"

def write_config(versions):
    cfg = {k: globals()[k] for k in ["METHOD_ID", "MODEL", "API_VERSION", "TEMPERATURE", "MAX_OUTPUT_TOKENS", "N_RUNS", "TEMPLATES",
                                     "TISSUE_CONTEXT", "TOP_GENES", "TOP_STATES", "LEGEND_GENES", "LETTER_SEED", "SLEEP_RANGE", "DRY_RUN", "IN_DIR"]}
    cfg.update({"model_versions_observed": sorted(versions), "decoding": "temperature and maxOutputTokens only; identical for all templates",
                "n_regions": len(letters), "n_states": len(sids), "leakage_check": "passed", "saved_at": datetime.datetime.now().isoformat(),
                "python": sys.version, "requests": requests.__version__, "pandas": pd.__version__})
    with open(f"{RESULTS_DIR}/run_config.json", "w") as f: json.dump(cfg, f, indent=2)

write_config([])
todo = [(t, r) for r in range(1, N_RUNS + 1) for t in TEMPLATES if not os.path.exists(raw_path(t, r))]
print("calls remaining:", len(todo))
if not DRY_RUN:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY_2")
    try:
        for t, r in todo:
            resp = call_gemini(PROMPTS[t], api_key)
            cand = (resp.get("candidates") or [{}])[0]
            parts = (cand.get("content") or {}).get("parts") or []
            text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
            fin = cand.get("finishReason") or ("BLOCKED:" + str(resp.get("promptFeedback", {}).get("blockReason", "")))
            with open(raw_path(t, r), "w") as f:
                json.dump({"template": t, "run": r, "temperature": TEMPERATURE, "finish_reason": fin, "text": text, "response": resp}, f)
            print("saved", t, r, fin)
            time.sleep(random.uniform(*SLEEP_RANGE))
    except StopRun as e:
        print("STOPPED:", e)
    left = [(t, r) for t, r in todo if not os.path.exists(raw_path(t, r))]
    print("remaining after stop:", left)

calls remaining: 20
saved T1 1 STOP
saved T2 1 STOP
saved T3 1 STOP
saved T4 1 STOP
saved T1 2 STOP
saved T2 2 STOP
saved T3 2 STOP
saved T4 2 STOP
saved T1 3 STOP
saved T2 3 STOP
saved T3 3 STOP
saved T4 3 STOP
saved T1 4 STOP
saved T2 4 STOP
saved T3 4 STOP
saved T4 4 STOP
saved T1 5 STOP
saved T2 5 STOP
saved T3 5 STOP
STOPPED: 429: {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/do
remaining after stop: [('T4', 5)]


In [5]:
def parse_answer(text):
    blocks = re.findall(r"```[a-zA-Z]*\s*\n?(.*?)```", text or "", flags=re.S)
    if not blocks: return None
    try:
        obj = json.loads(blocks[-1].strip())
    except json.JSONDecodeError:
        return None
    if not isinstance(obj, list) or not all(isinstance(o, dict) and "region" in o for o in obj): return None
    return {str(o["region"]).strip().upper(): o for o in obj}

COLS = ["method", "template", "run", "model", "model_version", "niche_letter", "niche_id", "n_cells", "label", "confidence",
        "justification", "parse_ok", "finish_reason", "raw_file"]
rows, versions = [], set()
for t in TEMPLATES:
    for r in range(1, N_RUNS + 1):
        p = raw_path(t, r)
        if not os.path.exists(p): continue
        raw = json.load(open(p)); ans = parse_answer(raw["text"]); mv = raw["response"].get("modelVersion", "")
        if mv: versions.add(mv)
        for L, nid, nn in zip(key["niche_letter"], key["niche_id"], key["n_cells"]):
            o = (ans or {}).get(L)
            rows.append({"method": METHOD_ID, "template": t, "run": r, "model": MODEL, "model_version": mv, "niche_letter": L,
                         "niche_id": nid, "n_cells": nn, "label": o.get("label", "") if o else "", "confidence": o.get("confidence", "") if o else "",
                         "justification": o.get("justification", "") if o else "", "parse_ok": o is not None,
                         "finish_reason": raw.get("finish_reason", ""), "raw_file": os.path.relpath(p, RESULTS_DIR)})
res = pd.DataFrame(rows, columns=COLS)
res.to_csv(f"{RESULTS_DIR}/results_{METHOD_ID}.csv", index=False)
write_config(versions)
print(len(res), "rows | parse_ok rate:", float(res["parse_ok"].mean()) if len(res) else None)

95 rows | parse_ok rate: 1.0


In [6]:
shutil.make_archive("/content/m1_prompt_results", "zip", RESULTS_DIR)
try:
    from google.colab import files
    files.download("/content/m1_prompt_results.zip")
except ImportError:
    print("zip saved at /content/m1_prompt_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>